# Whole-QM9 atomic environments at 6 Bohr, and the environment folds

Companion of `qm9_environment_clustering.py` (SOAP → per-element PCA → HDBSCAN over all 128,908 QM9 molecules →
`qm9_env_labels_6b.pkl.gz`) and `qm9_environment_folds.py` (five co-occurrence-closure holdouts → `qm9_env_folds_6b.json`).
The shipped files define the splits: the expensive steps run only with `RECOMPUTE = True` (on a node with `cache_6b/`)
and write `_rerun` copies.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import qm9_environment_clustering as qec      # this directory
import qm9_environment_folds as qef

RECOMPUTE = False                               # True: SOAP -> PCA -> HDBSCAN from cache_6b/ (hours without the cache)
CACHE_DIR = os.path.join(qec.SCRIPT_DIR, 'cache_6b')
LABELS_FILE = os.path.join(qec.SCRIPT_DIR, 'qm9_env_labels_6b.pkl.gz')               # shipped
FOLDS_FILE = os.path.join(qec.SCRIPT_DIR, 'qm9_env_folds_6b.json')                   # shipped
RERUN_FILE = os.path.join(CACHE_DIR, 'qm9_env_labels_6b_rerun.pkl.gz')               # RECOMPUTE output

## 1. Classification

In [ ]:
frame = qec.load_qm9_classification_frame(os.path.join(qec.MOLECULAR_DIR, 'aimel_clustered_molecular.pkl'),
                                          os.path.join(qec.MOLECULAR_DIR, 'qm9_filtered.pkl'),
                                          os.path.join(qec.MOLECULAR_DIR, 'qm9_classification_order.json.gz'))
print(f"{len(frame):,} molecules ({int(frame['in_aimel'].sum()):,} AIMEl), "
      f"{int(frame['atom'].map(len).sum()):,} atoms")

### 1.2 SOAP → PCA → HDBSCAN

SOAP at 6 Bohr (n_max 8, l_max 6, σ 0.7 Bohr; H/C/N/O), per-element PCA to 99 % variance (H 48 / C 76 / N 67 / O 52
components), HDBSCAN with `qec.COMMITTED`. 6 Bohr is close to what the molecular models perceive: their 4 hops on
3.5-Bohr graphs reach 98.3 % of each atom's 6-Bohr sphere.

In [ ]:
# Receptive field: atoms of each 6-Bohr sphere reached by 4 hops on the molecular graph
# (each atom's 4 closest neighbours within 3.5 Bohr, union, as representations.row_to_complex).
sample = frame[frame['in_aimel']].sample(3000, random_state=0)
share, whole = [], []
for _, row in sample.iterrows():
    xyz = np.stack([row['position_x'], row['position_y'], row['position_z']], axis=1).astype(float)
    d = np.linalg.norm(xyz[:, None] - xyz[None], axis=-1)
    np.fill_diagonal(d, np.inf)
    adj = np.zeros(d.shape, dtype=bool)
    np.put_along_axis(adj, np.argsort(d, axis=1)[:, :4], True, axis=1)
    adj = (adj | adj.T) & (d <= 3.5)
    hops = np.eye(len(d), dtype=bool)
    for _ in range(4):
        hops = hops | (hops @ adj)
    sphere, hit = d <= 6, hops & (d <= 6)
    has = sphere.any(1)
    share.extend(hit.sum(1)[has] / sphere.sum(1)[has])
    whole.extend(hit.sum(1) == sphere.sum(1))
print(f"4 hops reach {100 * np.mean(share):.1f} % of each atom's 6-Bohr sphere; "
      f"{100 * np.mean(whole):.1f} % of atoms have the whole sphere reached")


In [ ]:
if RECOMPUTE:
    meta = qec.stream_soap_to_memmaps(frame, CACHE_DIR, qec.soap_params(qec.RADIUS_BOHR))
    for el in qec.ELEMENTS:
        _, k, var = qec.pca_project(el, meta, CACHE_DIR)
        print(f'{el}: {k} PCA components ({var:.2f} % of the variance)')
    qec.commit(frame, meta, CACHE_DIR, qec.ELEMENTS, qec.COMMITTED, RERUN_FILE)
    rerun = pd.read_pickle(RERUN_FILE)
    shipped = pd.read_pickle(LABELS_FILE)
    same = [list(a) == list(b) for a, b in zip(rerun['atom_env_labels_6b'], shipped['atom_env_labels_6b'])]
    print(f"re-run vs shipped: identical labels for {100 * np.mean(same):.2f} % of the molecules")
else:
    print('RECOMPUTE = False: using the shipped labels')

### 1.3 Label sizes

In [ ]:
labels = pd.read_pickle(LABELS_FILE)
atoms = labels[['index', 'in_aimel', 'atom_env_labels_6b']].explode('atom_env_labels_6b', ignore_index=True)
atoms = atoms.rename(columns={'atom_env_labels_6b': 'label'})
atoms['element'] = atoms['label'].str.split('_').str[0]
atoms['set'] = np.where(atoms['in_aimel'].astype(bool), 'AIMEl', 'QM9 remainder')
counts = atoms.groupby(['element', 'label', 'set']).size().unstack('set', fill_value=0)
summary = pd.DataFrame({
    'labels': counts[~counts.index.get_level_values('label').str.endswith('_-1')].groupby('element').size(),
    'noise_%': atoms.assign(noise=atoms['label'].str.endswith('_-1')).groupby('element')['noise'].mean() * 100,
    'remainder_only_labels': counts[(counts['AIMEl'] == 0)].groupby('element').size()}).reindex(qec.ELEMENTS)
summary.fillna(0).round(1)

In [ ]:
# ── style of the figures (QT-Net's published figures: analysis/inference/analyse_inference.py) ──
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Liberation Sans'],
                     'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
FONTS = {'title': 18, 'label': 16, 'tick': 12, 'legend': 12}
TITLE_WEIGHT, LABEL_WEIGHT = 'normal', 'normal'
SET_COLORS = {'AIMEl': '#1a4f9c', 'QM9 remainder': '#e8604c'}    # as in the t-SNE maps of analysis/molecular

TITLE_FMT = '{element}: {n} labels, {noise:.1f} % noise'
XLABEL, YLABEL = 'Label (sorted by size)', 'Share of the set\'s atoms (%)'

fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout='constrained')
for ax, el in zip(axes.ravel(), qec.ELEMENTS):
    c = counts.loc[el]
    c = c[~c.index.str.endswith('_-1')]
    c = c.loc[c.sum(axis=1).sort_values(ascending=False).index]
    share = 100 * c / atoms[atoms['element'] == el].groupby('set').size()
    floor = 0.5 * share[share > 0].min().min()
    x = np.arange(len(share))
    for st, col in SET_COLORS.items():
        v = share[st].to_numpy()
        ax.scatter(x[v > 0], v[v > 0], s=22, color=col, label=st, zorder=3)
        ax.scatter(x[v == 0], np.full((v == 0).sum(), floor), s=40, facecolors='none', edgecolors=col, zorder=3)
    ax.set_yscale('log')
    ax.set_xticks([])
    ax.tick_params(labelsize=FONTS['tick'])
    ax.grid(alpha=0.3, lw=0.5)
    ax.set_title(TITLE_FMT.format(element=el, n=len(share), noise=summary.loc[el, 'noise_%']),
                 fontsize=FONTS['title'], fontweight=TITLE_WEIGHT)
    ax.set_xlabel(XLABEL, fontsize=FONTS['label'], fontweight=LABEL_WEIGHT)
    if el in ('H', 'N'):
        ax.set_ylabel(YLABEL, fontsize=FONTS['label'], fontweight=LABEL_WEIGHT)
axes[0, 0].legend(frameon=False, fontsize=FONTS['legend'])
plt.show()

**Table of the SI** (HDBSCAN details of the 6-Bohr clustering).

In [ ]:
PCA = {'H': (48, 99.01), 'C': (76, 99.01), 'N': (67, 99.01), 'O': (52, 99.02)}     # components, % variance retained
n_atoms = atoms.groupby(['element', 'set']).size().unstack('set')
paper_table = pd.DataFrame({
    'Number of atoms (AIMEl)': n_atoms['AIMEl'],
    'Number of atoms (QM9 remainder)': n_atoms['QM9 remainder'],
    'PCA components': {el: PCA[el][0] for el in qec.ELEMENTS},
    'Variance retained (%)': {el: PCA[el][1] for el in qec.ELEMENTS},
    'Min. cluster size': {el: qec.COMMITTED[el][0] for el in qec.ELEMENTS},
    'Min. samples': {el: qec.COMMITTED[el][1] for el in qec.ELEMENTS},
    'Number of clusters': summary['labels'],
    'Clusters without AIMEl atoms': summary['remainder_only_labels'],
    'Noise points (%)': summary['noise_%'].round(1)}).reindex(qec.ELEMENTS).fillna(0).astype(object).T
paper_table

## 2. Environment folds

Re-derived from the shipped labels and compared with `qm9_env_folds_6b.json`. How the cross-validation pairs them with
the chemotype groups: README, *Environment-Holdout Molecular Study*.

In [ ]:
design_atoms = qef.load_atoms(LABELS_FILE)
n_mol = design_atoms['mol_pos'].nunique()
folds = qef.design_folds(design_atoms, n_folds=5, cooc_thresh=0.95, max_mol_frac=0.25, min_label_atoms=50,
                         seed_frac_lo=0.01, seed_frac_hi=0.15, o_frac_lo=0.01, o_frac_hi=0.15)
fold_table = pd.DataFrame([{'fold': i, 'seed': f['seed'], 'labels': ', '.join(f['labels']),
                            'cooc H/C/N/O': '/'.join(f"{f['cooc'][e]:.2f}" for e in qec.ELEMENTS),
                            'AIMEl molecules': f['n_mols'], 'AIMEl %': round(100 * f['n_mols'] / n_mol, 1),
                            'atoms with the labels': f['n_eval_atoms'], 'overlap with earlier folds': f['overlap_with_prev']}
                           for i, f in enumerate(folds)]).set_index('fold')
display(fold_table)

shipped = json.load(open(FOLDS_FILE))['folds']
mol2qm9 = design_atoms.drop_duplicates('mol_pos').set_index('mol_pos')['qm9_index']
for f, s in zip(folds, shipped):
    assert f['labels'] == s['labels'], (f['labels'], s['labels'])
    assert sorted(int(q) for q in mol2qm9.loc[sorted(f['mols'])]) == s['qm9_index']
print(f'the re-derived folds equal {os.path.basename(FOLDS_FILE)} (labels and held-out molecules)')